# The user story — 10×10 array: lift 10 µm, traverse, drop

> *"Move this 10×10 atom array from A to B, lifting 10 µm out of plane on the way."*

That sentence is the whole product brief (`docs/PLAN.md` §1), and this notebook runs it end to
end: five lines of spec in, **RF waveforms for the four AOD channels** and a **simulated movie of
the tweezers** out, with every Table I number checked on the way through.

| Stage | What comes out |
|-------|----------------|
| `TrajectorySpec` → `synthesize` (Eq. S19) | 22 tones on four channels, band-checked against Eq. 1 |
| `WaveformSet.save` | a **parametric** NPZ — segments and coefficients, never samples |
| `simulate` → `SpotMetrics` | $(X, Y, \bar Z, \Delta F)$ per trap per frame, in closed form |
| `render_movie` | the headline artifact: 100 tweezers lifting, traversing and dropping |

Physics reference: arXiv:2510.11451 (equations `S#` refer to its Supplement); notebook 03 derives
the four-channel physics this page uses.

In [ ]:
import json
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aodl import (
    ArraySpec,
    Lift,
    TrajectorySpec,
    Translate,
    WaveformSet,
    auto_grid,
    default_1030,
    f_z_ramp,
    max_z_integral,
    render_movie,
    simulate,
    synthesize,
)
from aodl.units import MHz, um, us
from aodl.waveform.export import DEFAULT_SAMPLE_RATE, sample_times

P = default_1030()                  # paper hardware at lambda = 1030 nm (docs/PLAN.md 1.5)
optics = P.optics
tau = P.channels["Ax"].transit_time
OUT = Path("outputs")               # examples/outputs/ - gitignored


def with_order(params, order):
    "The same hardware with a different weak-drive expansion order (params.py)."
    return replace(params, channels={name: replace(a, mixing_order=order)
                                     for name, a in params.channels.items()})


P1 = with_order(P, 1)               # linear model for the sweep; order 3 spot-checked below

# ---- the story, in five lines ------------------------------------------------------------
story = TrajectorySpec(
    array=ArraySpec(10, 10, delta_f_x=1.0 * MHz, delta_f_y=1.3 * MHz),
    moves=(Lift(10 * um, 25 * us), Translate(40 * um, 25 * um, 30 * us), Lift(-10 * um, 25 * us)),
)
wfs = synthesize(story, P1)         # Eq. S19; raises if the drive leaves the band (Eq. 1)
# ------------------------------------------------------------------------------------------

pitch_x, pitch_y = story.array.pitch(P)
print(f"{story.array.n_traps} traps at {pitch_x / um:.2f} x {pitch_y / um:.2f} um pitch "
      f"-> array {9 * pitch_x / um:.0f} x {9 * pitch_y / um:.0f} um")
print(f"tones: " + ", ".join(f"{n}:{cw.n_tones}" for n, cw in wfs.channels.items())
      + f"  ({wfs.n_tones} total)")
print(f"trajectory {story.duration / us:.0f} us + {(wfs.t_span[1] - story.duration) / us:.1f} us hold "
      f"= {wfs.t_span[1] / us:.1f} us programmed   ({wfs.t_span[1] / tau:.1f} aperture transits)")
print(f"description: {wfs.description}")

## 1. Why the move is *fast*: Eq. 1 writes the schedule

The durations above are not a stylistic choice — they are what the RF band leaves. Holding the
array at $Z$ costs **every** channel a permanent chirp $\dot f_Z = Z/(2\,\text{lens\_scale})$
(48.5 MHz/ms at 10 µm), so the reachable axial "area" is

$$\Big|\int Z\,dt\Big| \;\le\; 2\,\text{lens\_scale}\,\big(f_{\max} - f_{\rm centre}\big) = 2.06\times10^{-9}\ \text{m·s}$$

*with nothing else in the band*. This array is not nothing: a 10-tone ladder at $\Delta f_x = 1$ MHz
occupies $\pm4.5$ MHz of `Bx`, the 1.3 MHz row spacing occupies $\pm5.85$ MHz of `By`, and the
lateral term of Eq. S19 adds $X/(2\,\text{deflection\_scale}) = 1.94$ MHz at the far end of the
traverse. What is left over for $f_Z$ is the budget below — about 60 µs of "$Z = 10$ µm time",
which is why the whole move takes 80 µs.

The unequal row/column spacings are deliberate for a second reason: with $\Delta f_x = \Delta f_y$
every anti-diagonal of the array shares one optical frequency $f_x + f_y$, and the traps of an
anti-diagonal would be reported as a single coherent group (`docs/conventions.md` §4).

Ask for the same move at a comfortable pace and the synthesizer refuses, with the arithmetic:

In [ ]:
comfortable = TrajectorySpec(
    array=story.array,
    moves=(Lift(10 * um, 150 * us), Translate(40 * um, 25 * um, 250 * us), Lift(-10 * um, 150 * us)),
)
try:
    synthesize(comfortable, P1)
except ValueError as exc:
    print(exc)

_, _, z_traj = story.compile()
f_z = f_z_ramp(z_traj, P)                       # the co-chirp every channel carries (Eq. S19)
ladder_x, ladder_y = story.array.detunings()
budget = {
    "Bx ladder (10 tones @ 1.0 MHz)": ladder_x.max(),
    "By ladder (10 tones @ 1.3 MHz)": ladder_y.max(),
    "lateral term, x (40 um / 2)": 40 * um / (2 * P.deflection_scale),
    "lateral term, y (25 um / 2)": 25 * um / (2 * P.deflection_scale),
    "f_Z, held after the drop": abs(float(f_z(story.duration))),
}
print("\nwhere the +10 MHz of headroom goes (By is the binding channel):")
for name, value in budget.items():
    print(f"  {name:32s} {value / MHz:6.2f} MHz")

t_scan = np.linspace(*wfs.t_span, 801)
reach = {name: float(np.max(np.abs(cw.eval_table(t_scan)["f"]))) for name, cw in wfs.channels.items()}
worst = max(reach, key=lambda name: reach[name])
print(f"  {'-> ' + worst + ' actually reaches':32s} {reach[worst] / MHz:6.2f} MHz of 10.00 "
      f"({100 * reach[worst] / (10 * MHz):.0f}% of the half-band)")
print(f"\nint Z dt = {2 * P.lens_scale * abs(float(f_z(story.duration))):.3e} m.s of the "
      f"{max_z_integral(P):.3e} m.s Eq. 1 ceiling, before the array and the traverse take theirs")

## 2. Deliverable 1 — the waveform file

`WaveformSet.save` writes the **parametric function representation** (`docs/PLAN.md` decision 5,
`docs/waveform_format.md`): per channel, one row per polynomial segment of each tone's frequency
law plus one row per tone for its phase and envelope, and a JSON blob with the hardware the
waveform was designed for. No samples — rendering to an AWG buffer is a separate step
(`aodl.waveform.export.render_samples`), and for this 103 µs move the two differ by ~60× in size.
The gap is not a constant: the sample file grows linearly with duration and sample rate, the
parametric one does not grow at all.

In [ ]:
npz = wfs.save(OUT / "04_array_move.npz")
with np.load(npz) as f:
    meta = json.loads(str(f["meta"]))
    contents = {key: f[key].shape for key in f.files if key != "meta"}
print(f"{npz}  ({npz.stat().st_size / 1e3:.1f} kB)")
print(f"  meta: schema {meta['schema_version']}, channels {meta['channels']}, "
      f"params snapshot with {len(meta['params'])} blocks")
for key, shape in contents.items():
    print(f"  {key:14s} {str(shape):10s}  " +
          ("(segment, t0, dt, degree, coeffs...)" if key.endswith("segments") else
           "(tone index, phase0, env kind, env params)"))

n_samples, _ = sample_times(wfs.t_span, DEFAULT_SAMPLE_RATE)
samples_bytes = n_samples * len(wfs.channels) * 4          # float32 per channel
print(f"\nsamples at {DEFAULT_SAMPLE_RATE / 1e6:.0f} MS/s: {n_samples:,} per channel x "
      f"{len(wfs.channels)} channels = {samples_bytes / 1e6:.2f} MB "
      f"({samples_bytes / npz.stat().st_size:.0f}x the parametric file)")

reloaded = WaveformSet.load(npz)
t_check = np.linspace(*wfs.t_span, 501)
worst = max(float(np.max(np.abs(reloaded.channels[n].eval_table(t_check)["f"]
                                - cw.eval_table(t_check)["f"])))
            for n, cw in wfs.channels.items())
print(f"round-trip: max |f_reloaded - f| = {worst:.2e} Hz")
assert worst == 0.0

## 3. Deliverable 2 — what the four channels actually do

Left: every tone's instantaneous frequency at the beam centre, $f_{\rm centre} + f(t - \tau/2)$,
the analytic version of the paper's Fig. 4b spectrogram — no FFT anywhere (`CLAUDE.md`). The two
`A` channels carry one tone each; the `B` channels carry the ten-tone ladders that *are* the
array. Every tone of every channel rides the same $f_Z$, which is why all four bundles drift
upward together during the lift and come back to a plateau (not to zero — the drive keeps the
frequency it has integrated) after the drop.

Right: how much of each channel's $\pm10$ MHz that costs. `By` is the binding constraint at 97%
of its band, and it is what sets the 80 µs schedule.

In [ ]:
t = np.linspace(*wfs.t_span, 400)
table = wfs.eval_table(t - 0.5 * tau)                 # what the beam centre sees (retarded)
colors = {"Ax": "#3a7bd5", "Bx": "#8ac926", "Ay": "#f4a261", "By": "#c1121f"}

fig, (ax_s, ax_b) = plt.subplots(1, 2, figsize=(12.0, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})
lo, hi = P.channels["Ax"].band
for name, color in colors.items():
    f_abs = P.channels[name].f_center + table[name]["f"]
    for row in f_abs:
        ax_s.plot(t / us, row / MHz, color=color, lw=1.1, alpha=0.85)
    ax_s.plot([], [], color=color, lw=1.4, label=f"{name} ({wfs.channels[name].n_tones} tones)")
for edge in (lo, hi):
    ax_s.axhline(edge / MHz, color="k", lw=0.9, ls="--")
for seam in np.cumsum([m.duration for m in story.moves]):
    ax_s.axvline(seam / us, color="k", lw=0.6, alpha=0.3)
ax_s.set(xlabel="t [µs]", ylabel="drive at the beam centre [MHz]", ylim=(88, 112),
         title="tone tracks: two ladders, four co-chirps (Eq. S19)")
ax_s.legend(fontsize=8, ncols=4, loc="lower right")

span = {name: (P.channels[name].f_center + table[name]["f"].min(),
               P.channels[name].f_center + table[name]["f"].max()) for name in colors}
for i, (name, (f0, f1)) in enumerate(span.items()):
    ax_b.barh(i, (f1 - f0) / MHz, left=f0 / MHz, color=colors[name], alpha=0.85)
    ax_b.text(f1 / MHz + 0.3, i, f"{100 * max(f1 - 100 * MHz, 100 * MHz - f0) / (10 * MHz):.0f}%",
              va="center", fontsize=8)
ax_b.axvspan(lo / MHz, hi / MHz, color="#8ac926", alpha=0.12)
for edge in (lo, hi):
    ax_b.axvline(edge / MHz, color="k", lw=0.9, ls="--")
ax_b.set(yticks=range(len(span)), yticklabels=list(span), xlabel="drive [MHz]", xlim=(88, 114),
         title="band usage (label: % of the half-band)")
plt.tight_layout()
plt.show()

## 4. Does it do what was asked?

`simulate` expands the drive into pupil terms and reduces them to one `SpotMetrics` per optical
frequency group — here 100 groups, one per trap — at each frame time. Every comparison is made
at the **retarded** time $t_c = t - \tau/2$: the acoustic sample that illuminates the beam centre
left the transducer half an aperture transit earlier, and v1 does not pre-compensate that
(`waveform/synthesis.py`). The four numbers that matter:

* **lateral** — the array centre against the requested $(X, Y)$, in waists;
* **axial** — $\bar Z$ against the requested $Z$, in Rayleigh ranges;
* **astigmatism** — $|\Delta F|$, which the paper's scheme holds at zero;
* **rigidity** — the spread of $\bar Z$ across the 100 traps.

In [ ]:
frames = np.linspace(tau, wfs.t_span[1], 120)          # the aperture is full from tau on
run = simulate(wfs, frames)
tab = run.spot_table()
x_req, y_req, z_req = story.compile()
t_c = frames - 0.5 * tau

centre = {key: np.array([np.mean([getattr(m, key) for m in frame]) for frame in run.metrics])
          for key in ("x", "y", "z_lab")}
spread = np.array([max(m.z_lab for m in frame) - min(m.z_lab for m in frame) for frame in run.metrics])

lateral = max(np.max(np.abs(centre["x"] - x_req(t_c))), np.max(np.abs(centre["y"] - y_req(t_c))))
axial = np.max(np.abs(centre["z_lab"] - z_req(t_c)))
astig = np.max(np.abs(tab["delta_f"]))
print(f"groups per frame           {len(run.metrics[0])} (= {story.array.n_traps} traps)")
print(f"worst lateral error        {lateral / optics.waist0:.2e} waists   ({lateral / um:.2e} um)")
print(f"worst axial error          {axial / optics.rayleigh:.2e} z_R      ({axial / um:.2e} um)")
print(f"worst |Delta F|            {astig / optics.rayleigh:.2e} z_R")
print(f"worst per-trap Z spread    {spread.max() / optics.rayleigh:.2e} z_R")
print(f"trap waists                {tab['wx'].min() / optics.waist0:.4f} .. {tab['wx'].max() / optics.waist0:.4f} w0")
assert lateral < 0.01 * optics.waist0
assert axial < 0.02 * optics.rayleigh
assert astig < 0.02 * optics.rayleigh
assert spread.max() < 0.02 * optics.rayleigh

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.6, 6.4))
(ax_xy, ax_z), (ax_t, ax_d) = axes

first, last = run.metrics[0], run.metrics[-1]
ax_xy.scatter([m.x / um for m in first], [m.y / um for m in first], s=6, color="#3a7bd5", label="t = tau")
ax_xy.scatter([m.x / um for m in last], [m.y / um for m in last], s=6, color="#c1121f", label="at rest")
ax_xy.plot(x_req(t_c) / um, y_req(t_c) / um, color="k", lw=1.2, alpha=0.5, label="requested centre")
ax_xy.set(xlabel="X [µm]", ylabel="Y [µm]", title="the array translates rigidly")
ax_xy.set_aspect("equal")
ax_xy.legend(fontsize=8, loc="lower right")

ax_z.plot(frames / us, z_req(t_c) / um, color="k", lw=3, alpha=0.25, label=r"requested $Z(t-\tau/2)$")
ax_z.plot(frames / us, centre["z_lab"] / um, color="#3a7bd5", lw=1.5, label=r"measured $\bar Z$")
ax_z.set(xlabel="t [µs]", ylabel="Z lab [µm]", title=r"$\bar Z$: up 10 µm and back down")
ax_z.legend(fontsize=8)

for label, values, color in (("X", centre["x"] - x_req(t_c), "#3a7bd5"),
                             ("Y", centre["y"] - y_req(t_c), "#f4a261")):
    ax_t.plot(frames / us, values / optics.waist0, color=color, lw=1.3, label=f"{label} error")
ax_t.plot(frames / us, (centre["z_lab"] - z_req(t_c)) / optics.rayleigh, color="#8ac926", lw=1.3,
          label=r"$\bar Z$ error [$z_R$]")
ax_t.set(xlabel="t [µs]", ylabel="tracking error", title="tracking error (waists / Rayleigh ranges)")
ax_t.legend(fontsize=8)

ax_d.plot(frames / us, np.abs(tab["delta_f"]).reshape(len(frames), -1).max(axis=1) / optics.rayleigh,
          color="#c1121f", lw=1.4, label=r"max $|\Delta F|$ over traps")
ax_d.plot(frames / us, spread / optics.rayleigh, color="#3a7bd5", lw=1.4, ls="--",
          label=r"per-trap $\bar Z$ spread")
ax_d.set(xlabel="t [µs]", ylabel=r"[$z_R$]", ylim=(-0.02, 0.02),
         title="astigmatism-free and rigid, to machine precision")
ax_d.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. One frame with the crystal's nonlinearity switched on

The sweep above ran at `mixing_order=1` — one tone, one beam — which is the right model for the
Eq. S19 *geometry* and keeps 120 frames of a 100-trap array cheap. The physical default is
`mixing_order=3`: expanding $e^{iCV}$ past first order mixes the ladder with itself and puts IM3
light at $f_j + f_k - f_i$ (Eqs. S20–S22, notebook 02). One frame at the product default shows
what that costs here — extra groups outside the array, and a per-mille redistribution inside it.

Two effects, both worth knowing before trusting a 100-trap number:

* **ghosts** — hundreds of weak groups appear around and inside the array grid (the products that
  fall outside the $\pm10$ MHz band are dropped: the transducer never launches them). Together
  they carry a few parts in $10^3$ of the array's light;
* **compression** — ten tones at modulation depth $m = CA = 0.3$ *each* is a deeply modulated
  crystal, and every trap lands at roughly half the small-signal intensity of the
  `mixing_order=1` model. That is a per-tone drive-strength statement, not a defect: it is what
  Eqs. S20–S22 predict, and it is why a real array trades per-tone depth against total RF power.

The Schroeder phases of Eq. S23 keep the residual per-trap intensity spread to a few percent
(notebook 02 §3 sweeps the alternatives).

In [ ]:
probe = 40 * us
mixed = simulate(synthesize(story, P), [probe]).metrics[0]
linear = simulate(wfs, [probe]).metrics[0]

scale = P.deflection_scale
ladder_x, ladder_y = story.array.detunings()
trap_x = x_req(probe - 0.5 * tau) + scale * ladder_x
trap_y = y_req(probe - 0.5 * tau) + scale * ladder_y
is_trap = np.array([
    min(abs(m.x - gx) for gx in trap_x) < 0.05 * optics.waist0
    and min(abs(m.y - gy) for gy in trap_y) < 0.05 * optics.waist0
    for m in mixed
])
power = np.array([m.power for m in mixed])
print(f"mixing_order=1: {len(linear):3d} groups   mixing_order=3: {len(mixed):3d} groups "
      f"({int(is_trap.sum())} traps + {int((~is_trap).sum())} ghosts)")
print(f"ghost light         = {power[~is_trap].sum() / power[is_trap].sum():.2e} of the array")
print(f"per-trap spread     = {power[is_trap].std() / power[is_trap].mean():.4f} (std/mean, Schroeder phases)")
print(f"array intensity vs order 1 = {power[is_trap].mean() / np.mean([m.power for m in linear]):.4f}")

fig, ax = plt.subplots(figsize=(5.6, 5.0))
sc = ax.scatter([m.x / um for m in mixed], [m.y / um for m in mixed],
                c=power / power[is_trap].mean(), norm="log", cmap="inferno", s=26)
ax.scatter([m.x / um for m, keep in zip(mixed, is_trap) if not keep],
           [m.y / um for m, keep in zip(mixed, is_trap) if not keep],
           facecolors="none", edgecolors="#3a7bd5", s=90, lw=1.0, label="IM3 ghosts")
ax.set(xlabel="X [µm]", ylabel="Y [µm]", title=f"groups at t = {probe / us:.0f} µs (mixing_order = 3)")
ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper left")
fig.colorbar(sc, ax=ax, shrink=0.8, label="power / mean trap")
plt.tight_layout()
plt.show()

## 6. The movie

The headline artifact: 100 tweezers whitening as they lift out of the focal plane, sliding to B,
and dropping back. The view is the package default `mode="tracked"` — the XY plane follows the
scene's own best focus — with **hue carrying each group's lab $Z$** (white in the plane, red
above) on one global brightness scale, an XZ slice through the array's row beside it, and the
four channel drives underneath with a cursor on the current frame.

The first $\tau$ = 11.5 µs is the startup transient of notebook 03: black until the
counter-propagating wavefronts meet at $\tau/2$, then a fast rise as the pairs fill.

Two knobs keep this inside a two-minute render: the frame count, and `xz_shape` — the XZ panel is
the one part of a frame that cannot be patched (a spot sweeps *through* focus along its Z axis),
so it costs $n_x n_z$ per frequency group, and this scene has a hundred of those.

In [ ]:
from IPython.display import Video

movie_frames = np.linspace(0.0, wfs.t_span[1], 96)
movie_run = simulate(wfs, movie_frames)
grid = auto_grid(movie_run, long_side=320)
print(f"grid {grid.nx} x {grid.ny} px over {(grid.x1 - grid.x0) / um:.0f} x {(grid.y1 - grid.y0) / um:.0f} um"
      f"  ({movie_run.n_frames} frames, tracked plane {movie_run.tracked_z().max() / um:.1f} um at the top)")

movie = render_movie(movie_run, OUT / "04_array_move.mp4", grid=grid, mode="tracked", fps=20,
                     xz_shape=(144, 96), spectrogram_panel=True, dpi=100)
print(f"{movie}  ({movie.stat().st_size / 1e3:.0f} kB)")
Video(str(movie), embed=True, html_attributes="controls loop")

## Where this stops, and what M4 adds

Everything above is one Eq. S19 chirp per channel, and that is exactly the limit: the schedule
was written by Eq. 1, not by the atoms. A 10 µm offset costs 48.5 MHz/ms on **all four** channels,
so this array — which has already spent 5.85 MHz of `By` on being an array — can stay up for
about 60 µs before the drive runs out of band. Longer holds, bigger arrays and wider traverses
all trade against the same 20 MHz.

Milestone 4 buys the axial degree of freedom back with **fading-Shepard** waveforms
(Eqs. S24–S28): a ladder per channel with $\cos^p$ fade envelopes, so a tone that reaches the band
edge hands over to a fresh one at the opposite edge and $\dot f_Z$ can be held indefinitely at
constant total intensity. The price is *shadow tweezers* at $\pm(\lambda F/v)\Delta f$ while two
tones overlap (Fig. S6) — which is why `field/focal.py` already groups terms by optical frequency
and adds degenerate ones coherently — and the fix is to interlace the $x$ and $y$ fading zones so
the shadows never coincide.

Until then, the honest summary of this page is the one the band-usage figure gives: **a 10×10
array, 10 µm out of plane and 47 µm across, in 80 µs, tracking the request to $10^{-13}$ waists
with $|\Delta F| < 10^{-15} z_R$ — and 97% of the RF band.**